# First Visualizations of Train Delays

In this notebook, we create the first visualizations based on the processed MMTIS data.

The goal is to explore delay patterns by:
- hour of day
- weekday
- station pairs

These visualizations will later become part of the final dashboard.

## 1. Import libraries

In [4]:
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

## 2. Load processed summary tables

In [5]:
from pathlib import Path

project_root = Path.cwd().parent

processed_path = project_root / "data" / "processed"

hour_summary = pd.read_csv(
    processed_path / "mmtis_hour_summary.csv"
)

weekday_summary = pd.read_csv(
    processed_path / "mmtis_weekday_summary.csv"
)

station_pair_summary = pd.read_csv(
    processed_path / "mmtis_station_pair_summary.csv"
)

heatmap_summary = pd.read_csv(
    processed_path / "mmtis_heatmap_summary.csv"
)

top_station_pairs = pd.read_csv(
    processed_path / "mmtis_top_station_pairs.csv"
)

In [6]:
hour_summary.head()

,departure_hour,mean_delay,median_delay,number_of_trips
0,0.0,1.608176,0.050000,9620
1,1.0,1.086015,-0.100000,2343
2,2.0,1.459279,0.083333,1383
3,3.0,2.771694,0.166667,2997
4,4.0,1.254605,0.266667,22593


In [7]:
weekday_summary.head()

,weekday,mean_delay,median_delay,number_of_trips
0,Friday,2.383747,0.600000,109559
1,Monday,2.093075,0.550000,108853
2,Saturday,1.350812,0.200000,93061
3,Sunday,1.303620,0.133333,84943
4,Thursday,2.188361,0.583333,108209


In [8]:
top_station_pairs.head()

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
0,PA,MLX,43.920531,30.800000,69
1,MLX,ND G,33.543229,14.991667,32
2,MKI,I W1,23.970000,24.733333,45
3,PAG,MLX,23.777624,13.950000,1048
4,HE,BC,23.306631,14.333333,93


## 3. Average delay by hour of day

This visualization shows how train delays change throughout the day.

We use the capped delay metric to reduce the impact of extreme outliers.

In [9]:
fig = px.line(
    hour_summary,
    x="departure_hour",
    y="mean_delay",
    markers=True,
    title="Average Delay by Hour of Day"
)

fig.show()

In [10]:
hour_summary.columns
weekday_summary.columns

Index(['weekday', 'mean_delay', 'median_delay', 'number_of_trips'], dtype='object')

## 4. Average delay by weekday

This chart shows average train delays for each day of the week.

It helps us identify whether some weekdays have systematically higher delays than others.

In [11]:
weekday_summary

,weekday,mean_delay,median_delay,number_of_trips
0,Friday,2.383747,0.600000,109559
1,Monday,2.093075,0.550000,108853
2,Saturday,1.350812,0.200000,93061
3,Sunday,1.303620,0.133333,84943
4,Thursday,2.188361,0.583333,108209
5,Tuesday,2.266212,0.566667,108937
6,Wednesday,2.120418,0.566667,110091


## 5. Average delay by weekday

This chart shows average train delays for each day of the week.

We compare weekdays and weekends to identify possible differences in train punctuality.

In [12]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_summary["weekday"] = pd.Categorical(
    weekday_summary["weekday"],
    categories=weekday_order,
    ordered=True
)

weekday_summary = weekday_summary.sort_values("weekday")

In [13]:
fig = px.bar(
    weekday_summary,
    x="weekday",
    y="mean_delay",
    title="Average Delay by Weekday"
)

fig.show()

### Observation

Train delays are generally higher during weekdays than during weekends.

Friday has the highest average delay, while Saturday and Sunday show the lowest delays.

This may indicate increased traffic and operational pressure during working days.

## 6. Delay heatmap by weekday and hour

This heatmap shows average train delays by:

- day of week
- hour of day

The goal is to identify periods with higher delay levels and possible congestion patterns.

In [15]:
heatmap_data = heatmap_summary.pivot(
    index="weekday",
    columns="departure_hour",
    values="mean_delay"
)

heatmap_data.head()

departure_hour,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,...,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0
weekday,,,,,,,,,,,,,,,,,,,,,
Friday,1.451728,0.979819,2.961111,3.125575,1.514890,2.069163,2.467257,2.538614,2.561764,2.242988,...,2.564842,2.647919,2.919531,2.637657,2.567389,2.491065,2.394825,1.824975,1.650766,1.236049
Monday,1.539865,4.444510,9.440000,3.343462,1.473088,1.971202,2.461188,2.610402,2.293539,2.113900,...,1.926957,2.150937,2.716842,2.294327,2.420457,2.153009,1.958182,1.708628,1.452445,1.621988
Saturday,1.692595,0.361044,0.372145,1.558601,0.750402,1.007457,1.098053,1.426409,1.538971,1.373735,...,1.340303,1.567325,1.671749,1.508668,1.600813,1.379206,1.441977,1.130122,1.073105,0.925530
Sunday,1.222658,0.444345,0.809202,2.065120,0.740245,0.943487,1.015414,1.053210,1.274027,1.025667,...,1.634984,1.524096,1.828861,1.534534,1.722053,1.442785,1.401592,1.309754,1.292436,1.245637
Thursday,1.791571,2.733926,2.849444,3.672348,1.204811,1.867591,2.429287,2.713906,2.576455,2.197626,...,2.199483,2.419289,2.994656,2.578751,2.468407,2.151363,2.072249,1.622405,1.410861,1.288021


In [16]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

heatmap_data = heatmap_data.reindex(weekday_order)

heatmap_data

departure_hour,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,...,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0
weekday,,,,,,,,,,,,,,,,,,,,,
Monday,1.539865,4.444510,9.440000,3.343462,1.473088,1.971202,2.461188,2.610402,2.293539,2.113900,...,1.926957,2.150937,2.716842,2.294327,2.420457,2.153009,1.958182,1.708628,1.452445,1.621988
Tuesday,1.952732,1.437751,3.522222,2.564815,1.305877,1.970966,2.366174,2.792292,2.617917,2.298615,...,2.227569,2.571075,2.994194,2.697336,2.358739,2.078442,2.067416,1.850752,1.537797,1.429615
Wednesday,1.682482,1.291667,14.152273,3.762251,1.306390,1.753087,2.295504,2.398126,2.275863,1.950655,...,2.190908,2.358602,3.025311,2.447955,2.342030,2.310486,2.222710,1.868699,1.456894,1.637179
Thursday,1.791571,2.733926,2.849444,3.672348,1.204811,1.867591,2.429287,2.713906,2.576455,2.197626,...,2.199483,2.419289,2.994656,2.578751,2.468407,2.151363,2.072249,1.622405,1.410861,1.288021
Friday,1.451728,0.979819,2.961111,3.125575,1.514890,2.069163,2.467257,2.538614,2.561764,2.242988,...,2.564842,2.647919,2.919531,2.637657,2.567389,2.491065,2.394825,1.824975,1.650766,1.236049
Saturday,1.692595,0.361044,0.372145,1.558601,0.750402,1.007457,1.098053,1.426409,1.538971,1.373735,...,1.340303,1.567325,1.671749,1.508668,1.600813,1.379206,1.441977,1.130122,1.073105,0.925530
Sunday,1.222658,0.444345,0.809202,2.065120,0.740245,0.943487,1.015414,1.053210,1.274027,1.025667,...,1.634984,1.524096,1.828861,1.534534,1.722053,1.442785,1.401592,1.309754,1.292436,1.245637


In [17]:
fig = px.imshow(
    heatmap_data,
    aspect="auto",
    title="Average Delay by Weekday and Hour",
    labels={
        "x": "Hour of Day",
        "y": "Weekday",
        "color": "Delay (min)"
    }
)

fig.show()

In [18]:
top_station_pairs.head()

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
0,PA,MLX,43.920531,30.800000,69
1,MLX,ND G,33.543229,14.991667,32
2,MKI,I W1,23.970000,24.733333,45
3,PAG,MLX,23.777624,13.950000,1048
4,HE,BC,23.306631,14.333333,93


## 7. Most delayed station pairs

This chart shows the station pairs with the highest average delay.

To avoid unreliable results, only station pairs with at least 30 observations were included.

In [22]:
top_station_pairs_plot = (
    top_station_pairs
    .head(15)
    .copy()
)

top_station_pairs_plot["station_pair"] = (
    top_station_pairs_plot["start_betriebsstelle"]
    + " → "
    + top_station_pairs_plot["ziel_betriebsstelle"]
)

top_station_pairs_plot.head()

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips,station_pair
0,PA,MLX,43.920531,30.800000,69,PA → MLX
1,MLX,ND G,33.543229,14.991667,32,MLX → ND G
2,MKI,I W1,23.970000,24.733333,45,MKI → I W1
3,PAG,MLX,23.777624,13.950000,1048,PAG → MLX
4,HE,BC,23.306631,14.333333,93,HE → BC


In [23]:
fig = px.bar(
    top_station_pairs_plot.sort_values("mean_delay"),
    x="mean_delay",
    y="station_pair",
    orientation="h",
    title="Top 15 Most Delayed Station Pairs"
)

fig.show()

### Note

The current visualization uses operational station codes from the MMTIS dataset.

In a later step, these codes will be mapped to human-readable station names using GTFS station information.